# Multi-Crop PlantVillage MobileNetV2 Training Pipeline (Google Colab GPU)

**Course:** Mini Project – I (CS-9130)  
**Student:** Anas Moinuddin Sayed  
**Hardware Target:** Google Colab T4 GPU (Free Tier - ~2-3 minutes total training time)

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Download and Extract Dataset in Colab
import os
import zipfile

print("Downloading PlantVillage dataset in Colab...")
!git clone --depth 1 https://github.com/ai-agriculture-circuits-and-systems/plant_village.git dataset_repo

DATASET_DIR = "dataset_repo/PlantVillage"
if not os.path.exists(DATASET_DIR):
    for root, dirs, files in os.walk("dataset_repo"):
        if len(dirs) > 30:
            DATASET_DIR = root
            break

print(f"Dataset located at: '{DATASET_DIR}'")

In [ ]:
# Step 3: Dependencies & Set Seed
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
MODELS_ASSETS_DIR = "models_assets"
MODEL_SAVE_PATH = os.path.join(MODELS_ASSETS_DIR, "mobilenet_v2_plantvillage.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_ASSETS_DIR, "class_names.json")
METRICS_SAVE_PATH = os.path.join(MODELS_ASSETS_DIR, "evaluation_metrics.json")

os.makedirs(MODELS_ASSETS_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
# Step 4: Create Native 3-Way Dataset Split (Train 70% / Val 15% / Test 15%)
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

class_names = raw_train_ds.class_names
num_classes = len(class_names)
print(f"Discovered {num_classes} classes.")

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

temp_batches = tf.data.experimental.cardinality(temp_ds).numpy()
val_batches = temp_batches // 2

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)

# Prefetch without .cache() to avoid RAM overflow on 56k images
train_ds = raw_train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Step 5: Compute Balanced Class Weights (Zero-RAM Overhead)
class_counts = []
for c_name in class_names:
    c_dir = os.path.join(DATASET_DIR, c_name)
    cnt = len([f for f in os.listdir(c_dir) if os.path.isfile(os.path.join(c_dir, f))]) if os.path.exists(c_dir) else 1
    class_counts.append(max(1, cnt))
total_samples = sum(class_counts)
class_weight_dict = {i: total_samples / (num_classes * count) for i, count in enumerate(class_counts)}
print('Balanced class weights computed for', len(class_weight_dict), 'classes.')

In [ ]:
# Step 6: Build MobileNetV2 Architecture with Exclusive Single Preprocessing
inputs = layers.Input(shape=(224, 224, 3), name='input_image')

x = layers.RandomFlip('horizontal_and_vertical', seed=SEED)(inputs)
x = layers.RandomRotation(0.2, seed=SEED)(x)
x = layers.RandomZoom(0.2, seed=SEED)(x)

# Exclusive MobileNetV2 Preprocessing Layer
x = layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input, name='mobilenet_v2_preprocess')(x)

base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3, seed=SEED)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

model = models.Model(inputs=inputs, outputs=outputs, name='MobileNetV2_PlantVillage')

In [ ]:
# Step 7: Phase 1 & Phase 2 Training on GPU (~2-3 mins total)
print('=== Phase 1: Training Classification Head ===')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)
model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight_dict, verbose=1)

print('=== Phase 2: Fine-Tuning Top Layers ===')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)
model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight_dict, verbose=1)

In [ ]:
# Step 8: Empirical Evaluation on Held-Out Test Set
test_loss, test_acc, test_top3_acc = model.evaluate(test_ds, verbose=1)

y_test_list = []
for _, labels in test_ds:
    y_test_list.extend(labels.numpy())
y_test = np.array(y_test_list)

test_preds = model.predict(test_ds, verbose=1)
y_pred = np.argmax(test_preds, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred).tolist()

print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}% | Top-3 Acc: {test_top3_acc*100:.2f}%')
print(f'F1 Score: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}')

metrics_payload = {
    'test_loss': round(float(test_loss), 4),
    'test_accuracy': round(float(test_acc * 100.0), 2),
    'test_top3_accuracy': round(float(test_top3_acc * 100.0), 2),
    'weighted_precision': round(float(precision), 4),
    'weighted_recall': round(float(recall), 4),
    'weighted_f1_score': round(float(f1), 4),
    'confusion_matrix': cm
}
with open(METRICS_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

model.save(MODEL_SAVE_PATH)
print(f"Model saved to '{MODEL_SAVE_PATH}'")

In [ ]:
# Step 9: Zip Assets for Direct Download back to Local Machine
import shutil
from google.colab import files

shutil.make_archive('models_assets', 'zip', 'models_assets')
files.download('models_assets.zip')
print('Downloaded models_assets.zip! Unzip into your project directory.')